In [ ]:
!pip install google-generativeai langchain langchain-core

In [1]:
import google.generativeai as genai
from getpass import getpass
from operator import itemgetter
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableSequence, RunnablePassthrough
from pprint import pprint

In [6]:
api_key = getpass("🔑 Enter your Gemini API key: ")
genai.configure(api_key=api_key)

# Create a custom LLM class that wraps Gemini for LangChain compatibility
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Mapping, Any

class GeminiLLM(LLM):
    model_name: str = "gemini-2.5-flash"
    
    @property
    def _llm_type(self) -> str:
        return "gemini"
    
    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        model = genai.GenerativeModel(self.model_name)
        response = model.generate_content(prompt)
        return response.text
    
    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {"model_name": self.model_name}

model = GeminiLLM()

## Define Each Component

In [3]:
# Prompt template – accepts a variable {topic}
prompt = ChatPromptTemplate.from_template("Write a short haiku about {topic}.")

# Parser
parser = StrOutputParser()

## Combine Using RunnableSequence
### Explicitly defining flow

In [4]:

chain = (
    prompt 
    | (lambda x: x.to_string())  # Regular function works fine with pipe operator
    | model 
    | parser
)

# Verify the type
print(f"Type of chain: {type(chain)}")

# Run it
result = chain.invoke({"topic": "autumn leaves"})
print(result)


Type of chain: <class 'langchain.schema.runnable.base.RunnableSequence'>
Golden, crimson hues,
Drift and dance upon the breeze,
Softly touch the ground.


## Inspect Intermediate Steps
### You can insert a RunnablePassthrough to peek at data as it flows.

In [7]:
# RunnablePassthrough lets data “pass through” unchanged
debug_chain = (
    prompt 
    | RunnablePassthrough()  # This will pass the data through unchanged
    | (lambda x: x.to_string())
    | model 
    | parser
)

# Try running it
result = debug_chain.invoke({"topic": "mountains"})
print(result)

Stone giants stand tall,
Silent watchers of the world,
Clouds embrace their peaks.


## Use itemgetter to Extract or Route Data

### When your inputs/outputs are dicts, itemgetter lets you pick pieces.

In [8]:
class DebugPassthrough(RunnablePassthrough):
    def invoke(self, input, config=None):
        pprint({"DEBUG INPUT": input})
        return super().invoke(input, config)

debug_chain = (
    prompt 
    | DebugPassthrough()
    | (lambda x: x.to_string())
    | model 
    | parser
)

result = debug_chain.invoke({"topic": "rain"})
print(result)

{'DEBUG INPUT': ChatPromptValue(messages=[HumanMessage(content='Write a short haiku about rain.', additional_kwargs={}, example=False)])}
Sky weeps gentle tears,
Thirsty earth drinks in the sound,
New life starts to bloom.


In [9]:
# A prompt needing both topic and mood
prompt = ChatPromptTemplate.from_template("Write a {mood} poem about {topic}.")

In [10]:
chain = (
    {
        "topic": itemgetter("topic"),
        "mood": itemgetter("mood")
    }
    | prompt
    | (lambda x: x.to_string())
    | model
    | parser
)

result = chain.invoke({"topic": "stars", "mood": "melancholic"})
print(result)

Oh, distant stars, you ancient gold,
A dust of light on velvet skies.
Your silent tales, so long untold,
Reflect a sadness in my eyes.

A ghostly glow from eons spun,
You might be ash, long turned to naught.
A journey finished, yet begun
In the lonely vision I have caught.

You watch us with an unblinking eye,
Our mortal sighs, our fleeting dream.
So silent, as our hopes all die,
Reflecting back no answering gleam.

We cast our burdens to your height,
Like tiny dust motes, lost and small.
You offer beauty, cold and bright,
But catch no whisper, hear no call.

And in that vast, black, endless deep,
Where galaxies like sorrows drift,
My human heart can only weep
For the unbridgeable, silent rift.

So shine, you stars, a haunting grace,
A cold comfort to my weary mind.
Reflecting back a lonely space,
Where solace I can never find.
